# Phase 5: Report Figures

This notebook produces the final, report-quality figures. Unlike the rough exploratory plots in Stage 2, these are built to directly illustrate the two halves of the hypothesis and the Stage 4 results:

1. **Stacked bar chart of error category proportions** by time pressure bin, per rating band -- shows the blunder tail growing under pressure (Claim 1, tail-driven).
2. **Box plots of `capped_cpl`** by time pressure bin, per rating band -- shows the median/bulk staying flat or shifting slightly down while the spread/outliers grow (the "bulk vs tail" distinction).
3. **Between-band line plot** of KS D-statistic and Cohen's d across transitions, one line per rating band -- the Stage 4.7 result as a picture (Claim 2, reversed trend).
4. **Pearson's r bar chart** by rating band -- the cleanest single-number visual for the "disproportionate effect" finding.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

RATING_BAND_LABELS = {
    1: 'Novice (<1000)',
    2: 'Intermediate (1000-1499)',
    3: 'Club Player (1500-1999)',
    4: 'Advanced (2000-2299)',
    5: 'Expert/Master (2300+)',
}
BAND_ORDER = [RATING_BAND_LABELS[b] for b in sorted(RATING_BAND_LABELS)]

TIME_PRESSURE_LABELS = {
    1: 'Minimal (>75%)',
    2: 'Low (50-75%)',
    3: 'Moderate (25-50%)',
    4: 'High (<25%)',
}

ERROR_CATEGORY_LABELS = {
    1: 'Inaccuracy (0-10)',
    2: 'Minor Error (11-50)',
    3: 'Major Error (51-150)',
    4: 'Blunder (151-300)',
}

TRANSITION_ORDER = ['Bin 1 -> Bin 2', 'Bin 2 -> Bin 3', 'Bin 3 -> Bin 4']

df = pd.read_csv('../../data/processed/analysed_moves.csv')
between_band = pd.read_csv('../results/between_band_comparison.csv', header=[0, 1], index_col=0)
pearson = pd.read_csv('../results/pearson_correlation_results.csv')

print(f'Loaded {len(df):,} rows')

## Figure 1: Error category composition by time pressure bin

For each rating band, this shows what proportion of moves fall into each error category (Inaccuracy / Minor / Major / Blunder), separately for each time pressure bin. If the "shift toward higher-severity errors" claim holds, the Blunder (and Major Error) slices should grow as time pressure increases (left to right within each panel), even if the Inaccuracy slice (the bulk) barely changes.

In [ ]:
category_colors = ['#4878a8', '#8fbcd4', '#f4a259', '#c44e52']

fig, axes = plt.subplots(1, 5, figsize=(18, 5), sharey=True)

for ax, rating_band in zip(axes, sorted(RATING_BAND_LABELS)):
    cell = df[df['rating_band'] == rating_band]
    proportions = (
        cell.groupby(['time_pressure_bin', 'error_category']).size()
        .unstack('error_category')
        .reindex(index=sorted(TIME_PRESSURE_LABELS), columns=sorted(ERROR_CATEGORY_LABELS))
    )
    proportions = proportions.div(proportions.sum(axis=1), axis=0)

    bottom = np.zeros(len(proportions))
    for cat, color in zip(sorted(ERROR_CATEGORY_LABELS), category_colors):
        ax.bar(proportions.index, proportions[cat], bottom=bottom, color=color,
               label=ERROR_CATEGORY_LABELS[cat], width=0.6)
        bottom += proportions[cat].values

    ax.set_title(RATING_BAND_LABELS[rating_band])
    ax.set_xticks(sorted(TIME_PRESSURE_LABELS))
    ax.set_xticklabels([TIME_PRESSURE_LABELS[b] for b in sorted(TIME_PRESSURE_LABELS)], rotation=45, ha='right')
    ax.set_xlabel('Time pressure bin')

axes[0].set_ylabel('Proportion of moves')
axes[-1].legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
fig.suptitle('Error category composition by time pressure bin, per rating band', y=1.02)
fig.tight_layout()
fig.savefig('../figures/error_category_composition.png', dpi=150, bbox_inches='tight')
plt.show()

## Figure 2: Box plots of capped CPL by time pressure bin

For each rating band, this shows the distribution of `capped_cpl` (median, IQR box, and whiskers) at each time pressure bin. Watch for the median line staying roughly flat (or dipping slightly) while the box and upper whisker stretch upward -- the visual signature of "the bulk doesn't move much, the tail gets heavier". Outlier points are hidden (`showfliers=False`) since with tens of thousands of points per box they would just form a solid smear; the spread is already captured by the whiskers (1.5 x IQR).

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 5), sharey=True)

for ax, rating_band in zip(axes, sorted(RATING_BAND_LABELS)):
    cell = df[df['rating_band'] == rating_band]
    data = [cell.loc[cell['time_pressure_bin'] == tp_bin, 'capped_cpl'] for tp_bin in sorted(TIME_PRESSURE_LABELS)]

    ax.boxplot(data, showfliers=False, labels=[TIME_PRESSURE_LABELS[b] for b in sorted(TIME_PRESSURE_LABELS)],
               medianprops={'color': '#c44e52', 'linewidth': 2})
    ax.set_title(RATING_BAND_LABELS[rating_band])
    ax.set_xticklabels([TIME_PRESSURE_LABELS[b] for b in sorted(TIME_PRESSURE_LABELS)], rotation=45, ha='right')
    ax.set_xlabel('Time pressure bin')

axes[0].set_ylabel('Capped CPL (centipawns)')
fig.suptitle('Capped CPL by time pressure bin, per rating band (outliers hidden)', y=1.02)
fig.tight_layout()
fig.savefig('../figures/cpl_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

## Figure 3: Between-band comparison -- KS D and Cohen's d across transitions

This is the Stage 4.7 result as a picture. Each line is one rating band; the x-axis is the transition (Bin 1->2, 2->3, 3->4), and the y-axis is the effect size. If the original hypothesis's "disproportionate effect on lower-rated players" were true, the Novice line would sit *above* the Expert/Master line, especially for later transitions. What we actually see is closer to the opposite for KS D, and for Cohen's d the Expert/Master line sits clearly apart from (and often opposite in sign to) the others.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for band_label in BAND_ORDER:
    ks_values = between_band.loc[band_label, ('KS_D', slice(None))].reindex(TRANSITION_ORDER, level=1)
    axes[0].plot(TRANSITION_ORDER, ks_values.values, marker='o', label=band_label)

    d_values = between_band.loc[band_label, ('cohens_d', slice(None))].reindex(TRANSITION_ORDER, level=1)
    axes[1].plot(TRANSITION_ORDER, d_values.values, marker='o', label=band_label)

axes[0].set_title("KS D-statistic by transition, per rating band")
axes[0].set_ylabel('KS D-statistic')
axes[0].set_xlabel('Transition')
axes[0].tick_params(axis='x', rotation=20)

axes[1].axhline(0, color='grey', linewidth=0.8, linestyle='--')
axes[1].set_title("Cohen's d by transition, per rating band")
axes[1].set_ylabel("Cohen's d")
axes[1].set_xlabel('Transition')
axes[1].tick_params(axis='x', rotation=20)
axes[1].legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)

fig.suptitle('Between-band comparison of effect sizes across time pressure transitions', y=1.02)
fig.tight_layout()
fig.savefig('../figures/between_band_effect_sizes.png', dpi=150, bbox_inches='tight')
plt.show()

## Figure 4: Pearson's r by rating band

One bar per rating band, showing the correlation between `time_remaining_pct` and `capped_cpl`. Because less time remaining should mean higher CPL, a negative r is the "expected direction" -- and the bars get more negative (further from zero) as rating band increases, again showing the effect strengthening with rating rather than weakening.

In [ ]:
ordered = pearson.set_index('rating_band_label').reindex(BAND_ORDER)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#c44e52' if r < 0 else '#4878a8' for r in ordered['pearson_r']]
ax.bar(ordered.index, ordered['pearson_r'], color=colors)
ax.axhline(0, color='grey', linewidth=0.8)
ax.set_ylabel("Pearson's r (time remaining % vs capped CPL)")
ax.set_xlabel('Rating band')
ax.set_xticklabels(ordered.index, rotation=20, ha='right')
ax.set_title("Correlation between time remaining and CPL, by rating band")
fig.tight_layout()
fig.savefig('../figures/pearson_r_by_band.png', dpi=150, bbox_inches='tight')
plt.show()